# Project imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from datetime import datetime
import time

from fontTools.merge.util import current_time

# initialize flappy game

In [2]:
import sys
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'
sys.path.insert(0, '../itml-project2')
# noinspection PyUnresolvedReferences
from ple.games.flappybird import FlappyBird
# noinspection PyUnresolvedReferences
from ple import PLE

pygame 2.6.1 (SDL 2.28.4, Python 3.13.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
couldn't import doomish


# ATC model


### Action space
- 0 -> Flap 119
- 1 -> Do nothing None

### State space
- player_y
- player_vel
- next_pipe_dist_to_player
- next_pip_top_y
- next_pipe_bottom_y
- next_next_pipe_dist_to_player
- next_next_pipe_top_y
- next_next_pipe_bottom_y

In [3]:
class PPO_Flappy(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(8,64)
        self.fc2 = nn.Linear(64,64)
        self.fc3 = nn.Linear(64,32)

        self.actor = nn.Linear(32,2) # actor outputs a probability of jumping or doing nothing
        self.critic = nn.Linear(32,1) # critic outputs value for state


    def forward(self,x):
        x = F.tanh(self.fc1(x)) # sigmoid activation function
        x = F.tanh(self.fc2(x)) # sigmoid activation function
        x = F.tanh(self.fc3(x))

        action_prob = F.softmax(self.actor(x),dim=-1) # return action probability
        state_value = self.critic(x) # Critic assessed state value
        return action_prob,state_value




# Hyper parameters

In [47]:
lr = 0.0003
epochs = 2000
K_epochs = 5

epsilon = 0.1 # Clipping range
gamma = 0.99 # Reward discount factor
lam = 0.95

# Loss Coeffs
c0 = 1
c1 = 0.01
c2 = 0.1


# Model initialization

In [48]:
model = PPO_Flappy()

try:
    model.load_state_dict(torch.load('flappy weights/FlappyModel2.pt'))
except FileNotFoundError:
    print("No weights found in: flappy weights/")


optimizer = torch.optim.Adam(model.parameters(),lr = lr)

No weights found in: flappy weights/


# Helper Functions

In [49]:
def normalize_game_state(state):
    means = torch.tensor([256.0, 0.0, 200.0, 200.0, 200.0, 400.0, 200.0, 200.0])
    stds = torch.tensor([128.0, 5.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0])
    return (state - means) / stds


def get_Advantage(states,rewards):
    # Calculating Advantage

    # V(S_t)
    with torch.no_grad():
        _,prev_values = model(states)

    temp = torch.cat([prev_values, torch.zeros(1, 1)], dim=0)

    G = 0
    advantages = []
    t = len(rewards) - 1
    for _, reward in enumerate(reversed(rewards)):
        delta = reward + gamma * temp[t + 1] - temp[t]
        G = delta + lam * gamma * G
        advantages.insert(0, torch.FloatTensor([G]))
        t -= 1

    advantages = torch.stack(advantages)
    A = advantages.squeeze()

    returns = A + prev_values.squeeze()
    return A, returns


# training Loop

In [50]:
import time

print_freq = 100

start_time = time.time()
game = FlappyBird()

total_rewards = []
all_L_clip = []
all_L_vf = []
all_L_entropy = []

for i in range(epochs):
    done = False
    p = PLE(game, fps=30, display_screen=False, force_fps=True)
    p.init()

    log_probs = []
    values = []
    rewards = []
    states = []
    agent_score = []
    actions = []

    while not done:
        game_state = normalize_game_state(torch.tensor(list(p.getGameState().values())))
        action_prob,critic_val = model(game_state) # get action and state from ATC

        # Actor Action
        action_prob = action_prob.detach().squeeze()
        dist = torch.distributions.Categorical(action_prob)
        action = dist.sample()

        # Critic State valuation
        critic_val = critic_val.detach().squeeze()

        if action == 0:
            ret_action = p.getActionSet()[0]
        else:
            ret_action = p.getActionSet()[1]

        # Acting on environment
        reward = p.act(ret_action)

        # Saving values
        log_probs.append(dist.log_prob(action))
        values.append(critic_val)
        rewards.append(reward)
        actions.append(action)
        states.append(game_state)

        #print_state(action,state,reward)
        if p.game_over():
            done = True
            p.reset_game()


    # Storing values after game loop
    total_rewards.append(sum(rewards))
    states = torch.stack(states).squeeze()
    values = torch.stack(values).squeeze()
    log_probs = torch.stack(log_probs).squeeze()
    actions = torch.stack(actions).squeeze()
    # Calculating advantage and storing returns for later
    A, returns = get_Advantage(states,rewards)


    for j in range(K_epochs):
        action_probs,new_values = model(states)
        new_dist = torch.distributions.Categorical(action_probs)
        new_policy = new_dist.log_prob(actions)

        ratio = torch.exp(new_policy-log_probs)
        clipped_ratio = torch.clamp(ratio,min = 1-epsilon,max= 1+epsilon)

        L_clip = (torch.min(ratio*A,clipped_ratio*A).mean())*c0
        L_vf = ((new_values.squeeze() - returns.squeeze())**2)*c1
        entropy_bonus = (new_dist.entropy())*c2

        loss = -(c0*L_clip-c1*L_vf+c2*entropy_bonus).mean()

        all_L_clip.append(L_clip.mean().item())
        all_L_vf.append(L_vf.mean().item())
        all_L_entropy.append(entropy_bonus.mean().item())


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    if (i+1) % print_freq == 0:

        current_time = time.time()
        elapsed_time = current_time - start_time

        epoch_num = i+1
        n_recent = int(K_epochs*print_freq)

        avg_reward = sum(total_rewards[-print_freq:]) /len(total_rewards[-print_freq:])
        L_clip_mean = sum(all_L_clip[-n_recent:]) / len(all_L_clip[-n_recent:])
        L_vf_mean = sum(all_L_vf[-n_recent:]) / len(all_L_vf[-n_recent:])
        L_entr_mean = sum(all_L_entropy[-n_recent:]) / len(all_L_entropy[-n_recent:])


        print(f"Epoch: {epoch_num}, "
              f"L_clip: {L_clip_mean:.2f}, "
              f"Squared loss: {L_vf_mean:.2f}, "
              f"Entropy loss: {L_entr_mean:.2f}, "
              f"Avg Rewards: {avg_reward:.2f}, "
              f"Time elapsed: {elapsed_time:.2f}s")

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


Epoch: 100, L_clip: -1.39, Squared loss: 0.04, Entropy loss: 0.07, Avg Rewards: -4.95, Time elapsed: 1.33s
Epoch: 200, L_clip: -1.26, Squared loss: 0.03, Entropy loss: 0.07, Avg Rewards: -4.88, Time elapsed: 2.64s
Epoch: 300, L_clip: -1.10, Squared loss: 0.03, Entropy loss: 0.06, Avg Rewards: -4.73, Time elapsed: 4.05s
Epoch: 400, L_clip: -1.09, Squared loss: 0.03, Entropy loss: 0.06, Avg Rewards: -4.80, Time elapsed: 5.44s
Epoch: 500, L_clip: -1.02, Squared loss: 0.02, Entropy loss: 0.06, Avg Rewards: -4.69, Time elapsed: 6.89s
Epoch: 600, L_clip: -1.00, Squared loss: 0.02, Entropy loss: 0.06, Avg Rewards: -4.74, Time elapsed: 8.34s
Epoch: 700, L_clip: -0.87, Squared loss: 0.02, Entropy loss: 0.05, Avg Rewards: -4.49, Time elapsed: 9.85s
Epoch: 800, L_clip: -0.83, Squared loss: 0.02, Entropy loss: 0.05, Avg Rewards: -4.55, Time elapsed: 11.36s
Epoch: 900, L_clip: -0.84, Squared loss: 0.02, Entropy loss: 0.05, Avg Rewards: -4.59, Time elapsed: 12.81s
Epoch: 1000, L_clip: -0.79, Squared

# Plotting Stats

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(range(len(total_rewards)), total_rewards, s=1, alpha=0.3, label='Episode reward')

# Rolling max reward
window = 100
if len(total_rewards) >= window:
    rolling_max = [max(total_rewards[max(0,i-window):i+1]) for i in range(len(total_rewards))]
    ax.plot(rolling_max, color='red', linewidth=1.5, label=f'Rolling max (window={window})')

ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.set_title(f'Training Rewards — Best: {max(total_rewards):.1f}')
ax.legend()
plt.tight_layout()
plt.show()

# Viewing model playing Flappy

In [ ]:
# Number of games to watch
import pygame
num_runs = 5

# Enable actual display (training set SDL_VIDEODRIVER to dummy)
os.environ.pop('SDL_VIDEODRIVER', None)
pygame.quit()
pygame.init()

model.eval()
game_vis = FlappyBird()
p_vis = PLE(game_vis, fps=30, display_screen=True, force_fps=False)
p_vis.init()

for run in range(num_runs):
    p_vis.reset_game()
    total_reward = 0
    while not p_vis.game_over():
        state = normalize_game_state(torch.tensor(list(p_vis.getGameState().values())))
        with torch.no_grad():
            action_prob, _ = model(state)
        action = (action_prob == max(action_prob))

        if action[0] == True:
            ret_action = p_vis.getActionSet()[0]
        else:
            ret_action = p_vis.getActionSet()[1]

        reward = p_vis.act(ret_action)
        total_reward += reward
    print(f"Run {run+1}/{num_runs} - Total reward: {total_reward:.1f}")

pygame.quit()
model.train()
print("Done.")

# Saving Model

In [ ]:
torch.save(model.state_dict(), "flappy weights/FlappyModel2 pt")